# 04 — Posterior Computation
**References:** Gelman et al. BDA3 Ch. 4 · Robert (2007) Ch. 4 · Gelman & Hill (2007)

## Narrative thread
```
Why computation? -> Grid approximation -> Laplace approximation -> Importance sampling -> Normalizing constant
```

## 1. The computational challenge

The posterior is:
$$p(\theta \mid \mathbf{x}) = \frac{p(\mathbf{x} \mid \theta)\, p(\theta)}{\int p(\mathbf{x} \mid \theta)\, p(\theta)\, d\theta}$$

The **marginal likelihood** (denominator) $p(\mathbf{x}) = \int p(\mathbf{x}\mid\theta)\,p(\theta)\,d\theta$ is a high-dimensional integral that is analytically intractable for most models.

**Four computational strategies:**

| Strategy | Works for | Scales to high $d$? |
|---|---|---|
| Grid approximation | $d \leq 3$ | No — exponential grid |
| Laplace approximation | Unimodal, regular posteriors | Yes |
| Importance sampling | Low to medium $d$ | Poorly |
| MCMC | General — workhorse | Yes (notebooks 05–06) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Grid approximation for Beta-Binomial ─────────────────────────────────
# X ~ Binomial(n,p),  p ~ Beta(2,2)  -- data: 6 successes in 15 trials
x_obs, n_obs = 6, 15
a0, b0 = 2, 2

p_grid      = np.linspace(0.001, 0.999, 1000)
log_prior   = stats.beta.logpdf(p_grid, a0, b0)
log_lik     = stats.binom.logpmf(x_obs, n_obs, p_grid)
log_post_un = log_prior + log_lik         # unnormalized log-posterior
post_un     = np.exp(log_post_un - log_post_un.max())  # stabilize
post_norm   = post_un / np.trapz(post_un, p_grid)      # normalize
post_analyt = stats.beta.pdf(p_grid, a0 + x_obs, b0 + n_obs - x_obs)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(p_grid, post_norm,   color='#4361ee', lw=2.5, label='Grid approx.')
axes[0].plot(p_grid, post_analyt, color='#f72585', lw=2, linestyle='--', label='Analytical')
axes[0].set_title(f'Grid approximation (1000 points)\nAnalytical: Beta({a0+x_obs},{b0+n_obs-x_obs})')
axes[0].set_xlabel('p'); axes[0].legend()

# Grid size comparison
grids = [10, 50, 200, 1000]
errors = []
for g in grids:
    pg = np.linspace(0.001, 0.999, g)
    lpu = stats.beta.logpdf(pg, a0, b0) + stats.binom.logpmf(x_obs, n_obs, pg)
    pu  = np.exp(lpu - lpu.max())
    pn  = pu / np.trapz(pu, pg)
    pa  = stats.beta.pdf(pg, a0+x_obs, b0+n_obs-x_obs)
    errors.append(np.max(np.abs(pn - pa)))

axes[1].loglog(grids, errors, 'o-', color='#4361ee', lw=2, markersize=8)
axes[1].set_xlabel('Grid size'); axes[1].set_ylabel('Max absolute error')
axes[1].set_title('Grid approximation error vs grid size\nError ~ O(1/grid_size)')

# 2D grid: Joint posterior for Normal (mu, sigma) -- small demo
data_2d = np.random.normal(2, 1.5, size=30)
mu_g    = np.linspace(-1, 5, 80)
sig_g   = np.linspace(0.5, 4, 80)
MU, SIG = np.meshgrid(mu_g, sig_g)

log_joint = np.zeros_like(MU)
for d in data_2d:
    log_joint += stats.norm.logpdf(d, MU, SIG)
log_joint += stats.norm.logpdf(MU, 0, 10)  # prior on mu
log_joint += stats.halfnorm.logpdf(SIG, scale=3)  # prior on sigma

joint = np.exp(log_joint - log_joint.max())
axes[2].contourf(MU, SIG, joint, levels=15, cmap='Blues')
axes[2].contour(MU, SIG, joint, levels=15, colors='white', linewidths=0.5, alpha=0.5)
axes[2].set_xlabel('$\\mu$'); axes[2].set_ylabel('$\\sigma$')
axes[2].set_title('2D grid: joint posterior for Normal$(\\mu,\\sigma)$\nGrid size = 80×80 = 6400 points')

plt.suptitle('Grid Approximation to the Posterior', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Laplace approximation

Approximate the posterior by a Gaussian centered at the MAP:

$$\log p(\theta \mid \mathbf{x}) \approx \log p(\hat\theta_{\text{MAP}} \mid \mathbf{x}) - \frac{1}{2}(\theta - \hat\theta)^\top \mathbf{H}^{-1}(\theta - \hat\theta)$$

where $\mathbf{H} = -\nabla^2 \log p(\theta \mid \mathbf{x})\big|_{\hat\theta}$ is the Hessian at the MAP.

This gives: $p(\theta \mid \mathbf{x}) \approx \mathcal{N}(\hat\theta, \mathbf{H}^{-1})$

**Quality:** Excellent when the posterior is unimodal and approximately normal (large $n$ by Bernstein-von Mises). Poor for multimodal or skewed posteriors.

In [ ]:
# ── Laplace approximation vs truth for skewed posteriors ──────────────────
# Model: X ~ Binomial(n,p),  p ~ Beta(1,1)
# Posterior: Beta(1+x, 1+n-x)  -- skewed for extreme x/n

configs = [(2, 30, 'Skewed: x=2, n=30'), (15, 30, 'Symmetric: x=15, n=30'), (28, 30, 'Skewed: x=28, n=30')]
p_g = np.linspace(0.001, 0.999, 500)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (x_c, n_c, title) in zip(axes, configs):
    a_post = 1 + x_c; b_post = 1 + n_c - x_c

    # True posterior
    true_post = stats.beta.pdf(p_g, a_post, b_post)

    # Laplace: MAP and Hessian
    def neg_log_post(p):
        if p <= 0 or p >= 1: return 1e10
        return -(stats.binom.logpmf(x_c, n_c, p) + stats.beta.logpdf(p, 1, 1))

    result  = optimize.minimize_scalar(neg_log_post, bounds=(0.01, 0.99), method='bounded')
    p_map   = result.x
    # Numerical Hessian (negative Hessian of log-posterior = precision)
    eps     = 1e-4
    hess    = (neg_log_post(p_map + eps) - 2*neg_log_post(p_map) + neg_log_post(p_map - eps)) / eps**2
    lap_sd  = 1 / np.sqrt(hess)
    laplace = stats.norm.pdf(p_g, p_map, lap_sd)

    ax.plot(p_g, true_post, color='#4361ee', lw=2.5, label='True posterior')
    ax.plot(p_g, laplace,   color='#f72585', lw=2, linestyle='--',
            label=f'Laplace $\\mathcal{{N}}({p_map:.2f},{lap_sd:.3f}^2)$')
    ax.set_title(f'{title}\nLaplace fails when posterior is skewed')
    ax.set_xlabel('p'); ax.legend(fontsize=8)

plt.suptitle('Laplace Approximation: Accuracy Depends on Posterior Shape', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Importance sampling

To compute $E_{p}[g(\theta)] = \int g(\theta)\,p(\theta\mid\mathbf{x})\,d\theta$ when $p$ is hard to sample:

1. Choose a **proposal distribution** $q(\theta)$ that is easy to sample
2. Draw $\theta^{(s)} \sim q(\theta)$
3. Compute **importance weights** $w^{(s)} = p(\theta^{(s)}\mid\mathbf{x}) / q(\theta^{(s)})$
4. Estimate: $E_p[g] \approx \frac{\sum_s w^{(s)} g(\theta^{(s)})}{\sum_s w^{(s)}}$

**Self-normalized IS:** Does not require knowing the normalizing constant $p(\mathbf{x})$ — only the unnormalized posterior $\tilde p(\theta) \propto p(\theta\mid\mathbf{x})$.

**Pareto-smoothed IS (PSIS-LOO):** Vehtari et al. — stabilizes extreme weights; used in LOO cross-validation (notebook 07).

**Diagnostic — effective sample size:**
$$n_{\text{eff}} = \frac{\left(\sum_s w^{(s)}\right)^2}{\sum_s (w^{(s)})^2}$$
Low $n_{\text{eff}}$ signals the proposal is a poor approximation to the posterior.

## 4. Key takeaways

| Method | Best for | Key limitation |
|---|---|---|
| Grid | $d \leq 3$, any shape | Exponential grid cost in $d$ |
| Laplace | Unimodal, large $n$ | Fails for skewed/multimodal posteriors |
| Importance sampling | Low $d$, diagnostic use | Weight collapse in high $d$ |
| MCMC | General Bayesian inference | Requires convergence diagnostics |

**Next:** notebook 05 — Metropolis-Hastings MCMC: theory, acceptance-rejection, convergence diagnostics.